# 05 — Mapping the per-capita crime rate

Same approach as London notebook 05 — `load_lad_boundaries()` is the same
generic function, just called with West Mercia's district set and
`name_column="District"`.

In [ ]:
import sys
sys.path.append("../../src")

import folium
import matplotlib.pyplot as plt
import seaborn as sns  # registers seaborn colormaps (e.g. "rocket") with matplotlib

from load_data import load_force_data, load_population, load_lad_boundaries
from clean import clean_crime_data, add_area_column, WEST_MERCIA_DISTRICTS

wm = load_force_data("west-mercia")
wm = clean_crime_data(wm)
wm = add_area_column(wm, column_name="District")
wmd = wm[wm["District"].isin(WEST_MERCIA_DISTRICTS)]

pop = load_population(WEST_MERCIA_DISTRICTS, name_column="District")
counts = wmd.groupby("District").size().rename("Crimes").reset_index()
merged = counts.merge(pop, on="District", validate="one_to_one")
merged["rate_per_1000"] = merged["Crimes"] / merged["Population"] * 1000

boundaries = load_lad_boundaries(WEST_MERCIA_DISTRICTS, name_column="District")
geo = boundaries.merge(merged, on="District", validate="one_to_one", how="left")
geo["rate_per_1000"].isna().sum()

## Chart — static choropleth map

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
geo.plot(
    column="rate_per_1000",
    cmap="rocket_r",
    linewidth=0.6,
    edgecolor="white",
    legend=True,
    legend_kwds={"label": "Crimes per 1,000 residents (annual)", "shrink": 0.6},
    ax=ax,
)
ax.set_title("Per-capita crime rate by West Mercia district")
ax.set_axis_off()
fig.tight_layout()

Worcester — tiny on the map, tucked between the larger districts — is the
one dark polygon, exactly matching the per-capita finding from notebooks
03-04. Its small physical size relative to Shropshire's or Herefordshire's
sprawling rural area is itself a visual hint at why: it's a compact city,
not a large rural county.

## Chart — interactive map

In [ ]:
# Same CRS reprojection trick as London's notebook 05: compute the centroid
# in a projected CRS (metres), then convert that one point back to
# lat/long for folium.
projected_centroids = geo.to_crs(epsg=27700).geometry.centroid
centroids = projected_centroids.to_crs(epsg=4326)
map_center = [centroids.y.mean(), centroids.x.mean()]

m = folium.Map(location=map_center, zoom_start=9, tiles="cartodbpositron")

folium.Choropleth(
    geo_data=geo.__geo_interface__,
    data=geo,
    columns=["District", "rate_per_1000"],
    key_on="feature.properties.District",
    fill_color="YlOrRd",
    fill_opacity=0.8,
    line_opacity=0.4,
    legend_name="Crimes per 1,000 residents (annual)",
).add_to(m)

folium.GeoJson(
    geo,
    style_function=lambda _: {"fillOpacity": 0, "color": "transparent"},
    tooltip=folium.GeoJsonTooltip(
        fields=["District", "rate_per_1000"],
        aliases=["District:", "Rate per 1,000:"],
    ),
).add_to(m)

m.save("../../outputs/west_mercia_crime_rate_map.html")
m

Also saved to `outputs/west_mercia_crime_rate_map.html` for viewing without
re-running the notebook.

## Where the West Mercia thread stands

This mirrors the full London progression (explore → clean → aggregate →
per-capita → deprivation → map) as an independent analysis, per the plan.
Both datasets are now ready for a genuine side-by-side comparison in
`notebooks/comparison/`.